# Datamine JOIN Process Example

This notebook demonstrates how to configure and run the **`join`** process wrapper in `dmstudio`.

## Process Description

## JOIN

Performs a relational join, subset join, weave or subset weave of two input files into an output file. The type of join required is controlled by the values of the optional parameters @**SUBSETR** AND @SUBSETF.

This process belongs to a group of four similar ones within the Datamine process collection; JOIN, SUBJOI, WEAVE and SUBWVE. Each provides a different outcome, as described by the following diagram:

[![](../Images/JOIN_SUBJOIN_WEAVE_SUBWEAVE.jpg)](<javascript:void\(0\);>)

By default, with @SUBSETR=0 and @SUBSETF=0, JOIN writes out all records and all fields from both input files. Records are compared on the specified keyfields, and if a match is found, the two records are combined into one on the output file. If both files have identical Data Definitions then the record from the second input file is the one written out. Thus in this case the second file updates the first. If no match is found then records are written out with absent data codes for the missing fields.

* With @SUBSETR=1 and @SUBSETF=0, a relational subset join is carried out. See process [SUBJOI](<subjoi.md>) for further details.

* With @SUBSETR=0 and @SUBSETF=1, a relational weave is carried out. See process [WEAVE](<weave.md>) for further details.

* With @SUBSETR=1 and @SUBSETF=1, a relational subset weave is carried out. See process [SUBWVE](<subwve.md>) for further details.

At least one keyfield must be specified and must appear in both input files as an explicit field. The keyfield may be up to 5 words long, and support is provided for up to 30 fields. If a field is specified which does not exist in both input files, it is ignored, providing at least one field matches.

Both input files must be sorted in the order of the key fields before they can be joined. If this is not the case, the process will exit with an error message.

### Input Files:

* **in1** (Table):
  First file to be updated (sorted on required keyfields).
  Required=Yes

* **in2** (Table):
  Second file (update file) (sorted on required keyfields).
  Required=Yes

### Output Files:

* **out** (Table):
  Output file.
  Required=Yes

### Fields:

### Parameters:

* **subsetr**:
  Controls whether all records or a subset are written to the output file. If set to (0)
  all records are written.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **subsetf**:
  Controls whether all fields or a subset is written to the output file. If set to (0)
  all fields are written. With SUBSETR and SUBSETF set to 0 JOIN writes out all records
  and all fields from both input files. With SUBSETR=1 and SUBSETF=0 a relational subset
  join is carried out. With SUBSETR=0 and SUBSETF=1 a relational weave is carried out.
  With SUBSETR=1 and SUBSETF=1 a relational subset weave is carried out.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **cartjoin**:
  If set to (0) and if no keyfields are specified the process will terminate with an
  error. If set to 1 the full Cartesian product is produced and written to the output
  file. No keyfields should be specified to produce the Cartesian product.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **keytol**:
  KEYTOL is the tolerance value used to test whether numeric key values are equal. It
  must be greater than or equal to zero. It replaces the previous heuristic comparison
  method. If KEYTOL is set to a negative value then zero is used. In a macro KEYTOL can be
  set to absent using -. "@KEYTOL=-" This will revert to legacy behaviour and use a
  heuristic comparison in relational commands and zero in sort.
  Range=0,+
  Values=Undefined
  Default=0.00001
  Required=No

* **print**:
  >=1 Display messages on Data definitions. Default is (0)
  Range=0,1
  Values=0,1
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('join')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute join
print("Running join...")
dm_cmd.join(
    inmods_i=['t_assays'],  # required
    out_o='t_join_out',  # required
    # subsetr_p=0,  # optional
    # subsetf_p=0,  # optional
    # cartjoin_p=0,  # optional
    # keytol_p=1e-05,  # optional
    # print_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("join execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_join_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")